In [8]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class RecyclingRobotEnv(gym.Env):
    """
    Custom Environment for the Recycling Robot (Sutton & Barto, Example 3.3).
    States: 0 = High, 1 = Low
    Actions: 0 = Search, 1 = Wait, 2 = Recharge
    """
    def __init__(self, alpha=0.8, beta=0.5, r_s=2.0, r_w=1.0):
        super(RecyclingRobotEnv, self).__init__()

        # Define Action and Observation Space
        self.action_space = spaces.Discrete(3)  # Search, Wait, Recharge
        self.observation_space = spaces.Discrete(2) # High, Low

        # MDP Parameters
        self.alpha = alpha  # Prob of staying High when searching in High
        self.beta = beta    # Prob of staying Low when searching in Low
        self.r_s = r_s      # Reward for searching
        self.r_w = r_w      # Reward for waiting
        self.r_rescue = -3.0 # Penalty for battery depletion

        self.state = 0 # Start in High state

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0 # Always start with high battery
        return self.state, {}

    def step(self, action):
        reward = 0
        terminated = False
        truncated = False

        if self.state == 0:  # HIGH BATTERY
            if action == 0:  # Search
                if np.random.random() < self.alpha:
                    self.state = 0 # Stays High
                else:
                    self.state = 1 # Goes to Low
                reward = self.r_s
            elif action == 1: # Wait
                self.state = 0 # Stays High
                reward = self.r_w
            elif action == 2: # Recharge (does nothing in High)
                self.state = 0
                reward = 0

        else:  # LOW BATTERY
            if action == 0:  # Search
                if np.random.random() < self.beta:
                    self.state = 1 # Stays Low
                    reward = self.r_s
                else:
                    self.state = 0 # Depleted! Rescued to High
                    reward = self.r_rescue
            elif action == 1: # Wait
                self.state = 1 # Stays Low
                reward = self.r_w
            elif action == 2: # Recharge
                self.state = 0 # Back to High
                reward = 0

        return self.state, reward, terminated, truncated, {}

In [9]:
def train_q_learning(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    # Initialize Q-table with zeros: (States x Actions)
    q_table = np.zeros([env.observation_space.n, env.action_space.n])

    for _ in range(episodes):
        state, _ = env.reset()
        done = False

        # Run for a fixed number of steps (infinite horizon task)
        for _ in range(100):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(q_table[state])

            next_state, reward, terminated, truncated, _ = env.step(action)

            # Q-Learning update rule
            old_value = q_table[state, action]
            next_max = np.max(q_table[next_state])

            new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
            q_table[state, action] = new_value

            state = next_state
            if terminated or truncated:
                break

    return q_table

# Execution
env = RecyclingRobotEnv()
q_table = train_q_learning(env)

# Display Results
states = ["High", "Low"]
actions = ["Search", "Wait", "Recharge"]

print("Learned Q-Table:")
print(q_table)

print("\nOptimal Policy:")
for s in range(len(states)):
    best_action = actions[np.argmax(q_table[s])]
    print(f"When battery is {states[s]}, the robot should: {best_action}")

Learned Q-Table:
[[16.94386772 16.06519756 15.112709  ]
 [13.97486871 14.56330255 15.03649798]]

Optimal Policy:
When battery is High, the robot should: Search
When battery is Low, the robot should: Recharge
